# 正解が分かっているデータを正しく判定できるのか
既存コードのデバッグをダミーデータを用いて行う

In [2]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
import nlopt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from scipy.optimize import minimize_scalar
from sklearn.model_selection import KFold
from scipy.optimize import brentq

from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

%load_ext autoreload
%autoreload 2

# 現在のフォルダの「1つ上の階層（../）」をシステムのパスに追加する
ROOT = Path(os.path.abspath("../.."))   # notebooks/check → notebooks → ルート
sys.path.append(str(ROOT))

#作成したモジュールをインポート
# from src.data_prep import *
# from src.nominal_speed_func import *
# from src.cooperation_surplus import *
# from src.macroscopic_data import *
# from src.traffic_regime import *
from src.whole_pipeline import *
from src.ring_equilibrium import *

#feet を　メートルに変換する係数 1feet=0.3048m
FT_TO_M = 0.3048

KM_TO_MILE = 1.60934

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
#パラメータセットを定義
#論文 Table 4（mph, vpm → km/h, veh/km に換算。θ1 は密度と同じ単位なので vpm→veh/km）
CAR_PARAMS = {'ub': 7.93 * KM_TO_MILE, 'uf': 73.55 * KM_TO_MILE, 'rhoc': 20.40 / KM_TO_MILE,
              'theta1': 8.0387 / KM_TO_MILE, 'theta2': 0.2309}
TRUCK_PARAMS = {'uf': 42.55 * KM_TO_MILE, 'rhoc': 41.74 / KM_TO_MILE}
A2_PAPER, B1_PAPER = 0.4528, 2.5996   #Table 5（無次元なので換算不要）

## 0. セットの定義

In [4]:
from src.ring_equilibrium import surplus_at

# a2, b1 のセット
SETS = {
    'P': (A2_PAPER, B1_PAPER),   # 論文の値 → どこでも s<0
    'T': (A2_PAPER, 1.0),        # テスト専用（b1 だけ中立の 1.0）→ どこでも s>0
    'N': (1.0, 1.0),             # 中立 → s=0 のはず
}
LAM_TRUE = 0.3    # L3 の正解の λ
EPS = 1e-6        # ε=0 の実装上の値 (km/h)。0 だと浮動小数点の誤差で 1-pipe にならない

## 0.5 正解を作る（src は使わない）
テストされる側（src）のコードで正解を作ると、同じバグがあっても一致してしまうので、ここでは src の関数を一切呼ばない。
使うのは論文のパラメータ（CAR_PARAMS, TRUCK_PARAMS, SETS）と定義の式だけ。

- 名目速度関数：論文 Eq.17, Eq.18 を書き直す。逆関数は閉じた式を使わず数値的に解く
- u*：Eq.7 は使わず、円環（1 km）の定常状態 Σ n_ij·h_ij(v) = 1 km（ランダム混合の平均）から解く
- p_i*, s：Eq.4, Definition 1
- regime：速度ではなく、道路シェア p と p* の比較で決める（混合 → 1-pipe、分離で p_i ≥ p_i* → 2-pipe、それ以外 → non-NE）

src の関数と名前がぶつからないように、関数名は `truth_` で始める

In [5]:
from scipy.optimize import brentq

# ---- 名目速度関数（自クラス）----
def truth_u_car(rho):
    """車：ロジスティック（論文 Eq.17）"""
    c = CAR_PARAMS
    return c['ub'] + (c['uf'] - c['ub']) / (1.0 + np.exp((rho - c['rhoc']) / c['theta1'])) ** c['theta2']


def truth_u_truck(rho):
    """トラック：アンダーウッド（論文 Eq.18）"""
    return TRUCK_PARAMS['uf'] * np.exp(-rho / TRUCK_PARAMS['rhoc'])


def truth_R_own(i, v):
    """速度 v で走るときの自クラスの密度 u_i^{-1}(v)。閉じた式を使わず数値的に解く"""
    u = truth_u_car if i == 1 else truth_u_truck
    return brentq(lambda r: u(r) - v, 0.0, 1000.0, xtol=1e-13)


# 確認：逆関数に通して戻すと元の速度になるか
v_test = 40.0
print(truth_u_car(truth_R_own(1, v_test)), truth_u_truck(truth_R_own(2, v_test)))

40.0 40.00000000000001


In [6]:
# ---- 1-pipe の正解速度（円環・ランダム混合の平均）----
def truth_1pipe_speed(rho1, rho2, a2, b1):
    """
    1 km の円環に ρ1 台の車と ρ2 台のトラックがランダムに混ざって並ぶ。
    ペア数: n_CC = ρ1 − m, n_CT = n_TC = m, n_TT = ρ2 − m,  m = ρ1·ρ2/ρ
    車間（車頭距離）: h_ij(v) = 1 / (a_ij · R_i(v))  [km]
    Σ n_ij · h_ij(v) = 1 km を v について解く
    """
    m = rho1 * rho2 / (rho1 + rho2)
    # (自車のクラス, 台数, スケーリング a_ij)
    pairs = [(1, rho1 - m, 1.0), (1, m, a2), (2, rho2 - m, 1.0), (2, m, b1)]

    def total_spacing_minus_L(v):
        return sum(n / (a * truth_R_own(i, v)) for i, n, a in pairs if n > 0) - 1.0

    lo = CAR_PARAMS['ub'] + 1e-6        # ロジスティックは ub より遅くならない
    hi = TRUCK_PARAMS['uf'] - 1e-6      # アンダーウッドは uf より速くならない
    return brentq(total_spacing_minus_L, lo, hi, xtol=1e-12)


# ---- 最小道路シェアと協力余剰（Eq.4, Definition 1）----
def truth_min_shares(rho1, rho2, u):
    p1s = rho1 / truth_R_own(1, u)
    p2s = rho2 / truth_R_own(2, u)
    return p1s, p2s, 1.0 - p1s - p2s


# ---- 分離して両クラスが同じ速度で走り、道路をちょうど使い切る速度（s(u) = 0 の u）----
def truth_separation_speed(rho1, rho2):
    f = lambda u: truth_min_shares(rho1, rho2, u)[2]
    return brentq(f, CAR_PARAMS['ub'] + 1e-6, TRUCK_PARAMS['uf'] - 1e-6, xtol=1e-12)


# ---- 正解の交通状態：道路シェア p と p* の比較で決める ----
def truth_regime(config, p1, p2, p1s, p2s, tol=1e-12):
    """
    混合 → 1-pipe（Li Corollary 1：1-pipe はどの密度でも NE）
    分離 → p1 ≥ p1* かつ p2 ≥ p2* なら 2-pipe（どちらも混ざるより速い）、そうでなければ non-NE
    （u_i(ρ_i/p_i) ≥ u* ⇔ p_i ≥ p_i* なので、Eq.11 の速度での判定と同じ条件）
    ※ p1 = p1*, p2 = p2* ちょうど（s = 0）の分離はケースに入れていない
    """
    if config == 'mixed':
        return '1-pipe'
    if p1 >= p1s - tol and p2 >= p2s - tol:
        return '2-pipe'
    return 'non-NE'

In [7]:
# ---- 1ケース分の正解を作る ----
def make_truth_case(case_id, set_name, rho1, rho2, config, label, split=None, note=''):
    """
    config: 'mixed'（ランダム混合の1車線）か 'separated'（車とトラックが別の車線）
    split : 分離のときのトラックの道路シェア p2 の決め方
            'lambda'          … p2 = p2* + (1−λ)s       （L3）
            'p2_star'         … p2 = p2*                 （トラックはちょうど u*、車が u* を下回る）
            'same_speed'      … 分離して両クラスが同じ速度 u_sep で走る分け方
            'p2_minus_half_s' … p2 = p2* − s/2           （トラックが u* を下回る）
    """
    a2, b1 = SETS[set_name]
    u_star = truth_1pipe_speed(rho1, rho2, a2, b1)
    p1s, p2s, s = truth_min_shares(rho1, rho2, u_star)

    if config == 'mixed':
        u1 = u2 = u_star                      # 全員が同じ速度
        p1 = p2 = np.nan
    else:
        if split == 'lambda':
            p2 = p2s + (1 - LAM_TRUE) * s
        elif split == 'p2_star':
            p2 = p2s
        elif split == 'same_speed':
            p2 = rho2 / truth_R_own(2, truth_separation_speed(rho1, rho2))
        elif split == 'p2_minus_half_s':
            p2 = p2s - 0.5 * s
        else:
            raise ValueError(f"split が不正です: {split}")
        p1 = 1.0 - p2
        u1, u2 = truth_u_car(rho1 / p1), truth_u_truck(rho2 / p2)

    regime = truth_regime(config, p1, p2, p1s, p2s)
    return {
        'case': case_id, 'set': set_name, 'rho1': rho1, 'rho2': rho2,
        'config': config, 'label': label, 'note': note,
        'u1_obs': u1, 'u2_obs': u2,                       # 正解の観測速度
        'u_star': u_star, 'p1_star': p1s, 'p2_star': p2s, 's': s,
        'p1': p1, 'p2': p2,
        'regime': regime,
        'coop_exists': (regime == '2-pipe') and (s > 0),
        'lambda_true': LAM_TRUE if split == 'lambda' else np.nan,
    }


# ---- 全22ケース ----
DENS = [(20.0, 2.0), (40.0, 4.0), (60.0, 6.0)]     # トラック約9%

def build_truth_cases():
    rows = []
    for k, (r1, r2) in enumerate(DENS, start=1):          # セットP（s<0）
        rows.append(make_truth_case(f'P-L1-{k}', 'P', r1, r2, 'mixed', 'L1', note='ランダム混合'))
    for k, (r1, r2) in enumerate(DENS, start=1):
        rows.append(make_truth_case(f'P-L4-{k}a', 'P', r1, r2, 'separated', 'L4', split='p2_star',
                                    note='p2 = p2* → 車が u* を下回る'))
        rows.append(make_truth_case(f'P-L4-{k}b', 'P', r1, r2, 'separated', 'L4', split='same_speed',
                                    note='分離して同速 u_sep < u* → 両方が下回る'))
    for k, (r1, r2) in enumerate(DENS, start=1):          # セットT（s>0、テスト専用）
        rows.append(make_truth_case(f'T-L2-{k}', 'T', r1, r2, 'mixed', 'L2', note='ランダム混合'))
    for k, (r1, r2) in enumerate(DENS + [(23.0, 15.5)], start=1):
        rows.append(make_truth_case(f'T-L3-{k}', 'T', r1, r2, 'separated', 'L3', split='lambda',
                                    note='p_i = p_i* + λs' + ('（s 最大の点。トラック40%で非現実的）' if k == 4 else '')))
    for k, (r1, r2) in enumerate(DENS, start=1):
        rows.append(make_truth_case(f'T-L4-{k}', 'T', r1, r2, 'separated', 'L4', split='p2_minus_half_s',
                                    note='p2 = p2* − s/2 → トラックが u* を下回る'))
    for k, (r1, r2) in enumerate(DENS, start=1):          # セットN（s=0 の確認）
        rows.append(make_truth_case(f'N-0-{k}', 'N', r1, r2, 'mixed', '(s=0)', note='a2=b1=1 → s=0 のはず'))
    return pd.DataFrame(rows)


cases = build_truth_cases()
cases[['case', 'set', 'rho1', 'rho2', 'config', 'label', 'u_star', 's', 'regime', 'coop_exists', 'note']]

,case,set,rho1,rho2,config,label,u_star,s,regime,coop_exists,note
0,P-L1-1,P,20.0,2.0,mixed,L1,60.025489,-2.539878e-01,1-pipe,False,ランダム混合
1,P-L1-2,P,40.0,4.0,mixed,L1,32.581564,-2.621954e-02,1-pipe,False,ランダム混合
2,P-L1-3,P,60.0,6.0,mixed,L1,19.424280,-1.172675e-02,1-pipe,False,ランダム混合
3,P-L4-1a,P,20.0,2.0,separated,L4,60.025489,-2.539878e-01,non-NE,False,p2 = p2* → 車が u* を下回る
4,P-L4-1b,P,20.0,2.0,separated,L4,60.025489,-2.539878e-01,non-NE,False,分離して同速 u_sep < u* → 両方が下回る
5,P-L4-2a,P,40.0,4.0,separated,L4,32.581564,-2.621954e-02,non-NE,False,p2 = p2* → 車が u* を下回る
6,P-L4-2b,P,40.0,4.0,separated,L4,32.581564,-2.621954e-02,non-NE,False,分離して同速 u_sep < u* → 両方が下回る
7,P-L4-3a,P,60.0,6.0,separated,L4,19.424280,-1.172675e-02,non-NE,False,p2 = p2* → 車が u* を下回る
8,P-L4-3b,P,60.0,6.0,separated,L4,19.424280,-1.172675e-02,non-NE,False,分離して同速 u_sep < u* → 両方が下回る
9,T-L2-1,T,20.0,2.0,mixed,L2,54.036412,6.675866e-02,1-pipe,False,ランダム混合


In [8]:
# ---- 正解の自己チェック ----
# 1) 求めた u* を印刷版 Eq.7 に直接代入すると 1 になるか（円環の式と Eq.7 が同じ式であることの確認）
res = []
for _, r in cases.iterrows():
    a2, b1 = SETS[r['set']]
    u, r1, r2 = r['u_star'], r['rho1'], r['rho2']
    lhs = (r1 / truth_R_own(1, u) * (r1 + r2 / a2) + r2 / truth_R_own(2, u) * (r1 / b1 + r2)) / (r1 + r2)
    res.append(abs(lhs - 1.0))
print(f"Eq.7 の残差の最大値: {max(res):.1e}")

# 2) セットN（中立）では s = 0 になるか
print(f"セットN の |s| の最大値: {cases.loc[cases['set'] == 'N', 's'].abs().max():.1e}")

# 3) 以前に作った CSV と同じ値か（CSV がなければ飛ばす）
csv_path = ROOT / "results" / "stage1_2026-11-07" / "stage1_cases.csv"
if csv_path.exists():
    old = pd.read_csv(csv_path).set_index('case')
    new_ = cases.set_index('case')
    num_cols = ['u1_obs', 'u2_obs', 'u_star', 'p1_star', 'p2_star', 's']
    print(f"CSV との差の最大値: {(new_[num_cols] - old.loc[new_.index, num_cols]).abs().max().max():.1e}")
    print(f"regime が CSV と一致: {(new_['regime'] == old.loc[new_.index, 'regime']).all()}")

Eq.7 の残差の最大値: 5.3e-15
セットN の |s| の最大値: 2.8e-16
CSV との差の最大値: 1.1e-14
regime が CSV と一致: True


In [14]:
# まずは最低限の7ケース。全22ケースでやるときは target = cases.copy() にする
MIN7 = ['P-L1-2', 'T-L2-2', 'T-L3-2', 'P-L4-2a', 'P-L4-2b', 'T-L4-2', 'N-0-2']
# target = cases[cases['case'].isin(MIN7)].reset_index(drop=True)
target = cases.copy()
target[['case', 'set', 'rho1', 'rho2', 'config', 'label', 'note']]

,case,set,rho1,rho2,config,label,note
0,P-L1-1,P,20.0,2.0,mixed,L1,ランダム混合
1,P-L1-2,P,40.0,4.0,mixed,L1,ランダム混合
2,P-L1-3,P,60.0,6.0,mixed,L1,ランダム混合
3,P-L4-1a,P,20.0,2.0,separated,L4,p2 = p2* → 車が u* を下回る
4,P-L4-1b,P,20.0,2.0,separated,L4,分離して同速 u_sep < u* → 両方が下回る
5,P-L4-2a,P,40.0,4.0,separated,L4,p2 = p2* → 車が u* を下回る
6,P-L4-2b,P,40.0,4.0,separated,L4,分離して同速 u_sep < u* → 両方が下回る
7,P-L4-3a,P,60.0,6.0,separated,L4,p2 = p2* → 車が u* を下回る
8,P-L4-3b,P,60.0,6.0,separated,L4,分離して同速 u_sep < u* → 両方が下回る
9,T-L2-1,T,20.0,2.0,mixed,L2,ランダム混合


## 1. 入力作成：観測速度を円環の定常状態から作る
- 混合：`true_1pipe_speed`（Σ n_ij·h_ij(v) = L、ランダム混合の平均）→ u1_obs = u2_obs
- 分離：`separated_speeds`（u_i(ρ_i / p_i)）。L3 は p2 = p2* + (1−λ)s、L4 は正解表（0.5）の p2

In [16]:
def make_observed_speed(case_row):
    """1ケース分の観測速度 (u1_obs, u2_obs) を作る"""
    a2, b1 = SETS[case_row['set']]
    nominal = make_paper_nominal(CAR_PARAMS, TRUCK_PARAMS, a2, b1)
    rho1, rho2 = case_row['rho1'], case_row['rho2']

    # ランダム混合（平均）の円環の定常速度
    v_mix = lane_speed(nominal, pair_counts(rho1, rho2), 1000.0)

    if case_row['config'] == 'mixed':
        return v_mix, v_mix                                  # 混合：全員が同じ速度

    # 分離：トラックの道路シェア p2 を決める（車は 1 - p2）
    if case_row['label'] == 'L3':
        r = surplus_at(nominal, rho1, rho2, v_mix)           # p1*, p2*, s
        p2 = r['p2_star'] + (1 - LAM_TRUE) * r['s']          # p2 = p2* + (1-λ)s
    else:
        p2 = case_row['p2']                                  # L4：正解表に書いた分け方をそのまま使う
    return separated_speeds(nominal, rho1, rho2, p2)


rows = []
for _, c in target.iterrows():
    u1, u2 = make_observed_speed(c)
    rows.append({'case': c['case'], 'set': c['set'], 'rho1': c['rho1'], 'rho2': c['rho2'],
                 'u1_obs': u1, 'u2_obs': u2})
df_in = pd.DataFrame(rows)

# 入力の段階で正解と合っているか
df_in['d_u1_obs'] = df_in['u1_obs'] - target['u1_obs']
df_in['d_u2_obs'] = df_in['u2_obs'] - target['u2_obs']
df_in

,case,set,rho1,rho2,u1_obs,u2_obs,d_u1_obs,d_u2_obs
0,P-L1-1,P,20.0,2.0,60.025489,60.025489,0.000000e+00,0.000000e+00
1,P-L1-2,P,40.0,4.0,32.581564,32.581564,7.105427e-15,7.105427e-15
2,P-L1-3,P,60.0,6.0,19.424280,19.424280,3.552714e-15,3.552714e-15
3,P-L4-1a,P,20.0,2.0,33.166908,60.025489,0.000000e+00,0.000000e+00
4,P-L4-1b,P,20.0,2.0,55.757182,55.757182,0.000000e+00,0.000000e+00
5,P-L4-2a,P,40.0,4.0,31.154991,32.581564,-1.065814e-14,0.000000e+00
6,P-L4-2b,P,40.0,4.0,31.600269,31.600269,0.000000e+00,0.000000e+00
7,P-L4-3a,P,60.0,6.0,19.111361,19.424280,0.000000e+00,0.000000e+00
8,P-L4-3b,P,60.0,6.0,19.163028,19.163028,0.000000e+00,0.000000e+00
9,T-L2-1,T,20.0,2.0,54.036412,54.036412,-1.421085e-14,-1.421085e-14


## 2. アルゴリズムを流す（Step 3a〜5）
src の関数をそのまま呼ぶだけ。src の中身は変えない

In [17]:
def run_algorithm(df_in):
    df = df_in[['case', 'set', 'rho1', 'rho2', 'u1_obs', 'u2_obs']].copy()

    # Step 3a: 1-pipe 速度 u*（印刷版 Eq.7）。a2, b1 がセットごとに違うので1行ずつ解く
    # （whole_pipeline 版の solve_1pipe_speed は scaling の指定が必須）
    u_star = []
    for _, r in df.iterrows():
        a2, b1 = SETS[r['set']]
        u_star.append(solve_1pipe_speed(r['rho1'], r['rho2'], CAR_PARAMS, TRUCK_PARAMS, a2, b1,
                                        scaling='printed'))
    df['u_1pipe'] = u_star

    df = compute_min_road_share(df, CAR_PARAMS, TRUCK_PARAMS)   # Step 3b: p1*, p2*
    df = compute_cooperation_surplus(df)                        # Step 3c: s
    df = classify_traffic_regime(df, eps_kmh=EPS)               # Step 4 : regime
    df = check_cooperation_existence(df)                        # Step 5 : coop_exists
    return df


df_out = run_algorithm(df_in)
df_out[['case', 'u_1pipe', 'p1_star', 'p2_star', 's', 'regime', 'coop_exists']]

最小道路占有を計算: 22スナップショット中、22件で計算完了
協力余剰sを計算: 22スナップショット中、22件で計算完了（うちs>0: 12件）
交通状態を判定(eps: 車 0.00, トラック 0.00 km/h): 判定対象 22件 (対象外 undefined: 0件)
      2-pipe: 4件 (18.18%)
      1-pipe: 9件 (40.91%)
      non-NE: 9件 (40.91%)
  Pcoop = 18.18% (論文: 13.83%)
  平均s: 2-pipe = 0.0985, 1-pipe = -0.0068 (論文: 0.15, 0.11)
協力性の存在判定: 判定対象 22件
  s>0             : 12件 (54.55%)
  2-pipe          : 4件 (18.18%)  ← 論文Pcoop=13.83%
  2-pipe かつ s>0 : 4件 (18.18%)  ← 協力性あり(Step6で使う)
  2-pipe かつ s<=0: 0件 (0.00% of 2-pipe)  ← 命題1と不整合


,case,u_1pipe,p1_star,p2_star,s,regime,coop_exists
0,P-L1-1,60.025489,0.668625,0.585363,-2.539878e-01,1-pipe,False
1,P-L1-2,32.581564,0.818581,0.207639,-2.621954e-02,1-pipe,False
2,P-L1-3,19.424280,0.828122,0.183604,-1.172675e-02,1-pipe,False
3,P-L4-1a,60.025489,0.668625,0.585363,-2.539878e-01,non-NE,False
4,P-L4-1b,60.025489,0.668625,0.585363,-2.539878e-01,non-NE,False
5,P-L4-2a,32.581564,0.818581,0.207639,-2.621954e-02,non-NE,False
6,P-L4-2b,32.581564,0.818581,0.207639,-2.621954e-02,non-NE,False
7,P-L4-3a,19.424280,0.828122,0.183604,-1.172675e-02,non-NE,False
8,P-L4-3b,19.424280,0.828122,0.183604,-1.172675e-02,non-NE,False
9,T-L2-1,54.036412,0.607660,0.325581,6.675866e-02,1-pipe,False


## 3. Step 6：λ の推定（L3 だけ）
`estimate_surplus_split` は10行未満だとエラーになるので、`fit_surplus_split` を直接呼ぶ

In [18]:
df_L3 = df_out[target['label'] == 'L3']

# 1枚ずつ推定
for case_id in df_L3['case']:
    lam = fit_surplus_split(df_L3[df_L3['case'] == case_id], CAR_PARAMS, TRUCK_PARAMS)
    print(f"{case_id}: λ̂ = {lam:.6f}   (正解 {LAM_TRUE}, 差 {lam - LAM_TRUE:+.2e})")

# まとめて推定（全22ケースのときは T-L3 の4枚）
lam_all = fit_surplus_split(df_L3, CAR_PARAMS, TRUCK_PARAMS)
print(f"まとめて: λ̂ = {lam_all:.6f}   (差 {lam_all - LAM_TRUE:+.2e})")

T-L3-1: λ̂ = 0.300000   (正解 0.3, 差 +2.48e-08)
T-L3-2: λ̂ = 0.300000   (正解 0.3, 差 -2.09e-08)
T-L3-3: λ̂ = 0.300000   (正解 0.3, 差 -1.68e-08)
T-L3-4: λ̂ = 0.300001   (正解 0.3, 差 +5.30e-07)
まとめて: λ̂ = 0.300000   (差 -5.83e-09)


## 4. 突き合わせ：正解（0.5 で作った表）と比べる
合格の目安：u* の差 < 1e-6 km/h、p*・s の差 < 1e-8、regime・coop_exists は完全一致

In [19]:
TOL_U = 1e-6     # 速度 (km/h)
TOL_P = 1e-8     # p*, s

truth = target.set_index('case')
out = df_out.set_index('case')

check = pd.DataFrame({
    'label':       truth['label'],
    'd_u_star':    out['u_1pipe'] - truth['u_star'],
    'd_p1_star':   out['p1_star'] - truth['p1_star'],
    'd_p2_star':   out['p2_star'] - truth['p2_star'],
    'd_s':         out['s'] - truth['s'],
    'regime':      out['regime'],
    'regime_true': truth['regime'],
    'coop':        out['coop_exists'],
    'coop_true':   truth['coop_exists'],
})
check['ok_u'] = check['d_u_star'].abs() < TOL_U
check['ok_p'] = (check[['d_p1_star', 'd_p2_star', 'd_s']].abs() < TOL_P).all(axis=1)
check['ok_regime'] = check['regime'] == check['regime_true']
check['ok_coop'] = check['coop'] == check['coop_true']
check['ALL_OK'] = check[['ok_u', 'ok_p', 'ok_regime', 'ok_coop']].all(axis=1)

print(f"合格: {check['ALL_OK'].sum()} / {len(check)} ケース")
check

合格: 22 / 22 ケース


,label,d_u_star,d_p1_star,d_p2_star,d_s,regime,regime_true,coop,coop_true,ok_u,ok_p,ok_regime,ok_coop,ALL_OK
case,,,,,,,,,,,,,,
P-L1-1,L1,0.000000e+00,0.000000e+00,2.220446e-16,-2.220446e-16,1-pipe,1-pipe,False,False,True,True,True,True,True
P-L1-2,L1,1.065814e-13,1.776357e-15,8.049117e-16,-2.581269e-15,1-pipe,1-pipe,False,False,True,True,True,True,True
P-L1-3,L1,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1-pipe,1-pipe,False,False,True,True,True,True,True
P-L4-1a,L4,0.000000e+00,0.000000e+00,2.220446e-16,-2.220446e-16,non-NE,non-NE,False,False,True,True,True,True,True
P-L4-1b,L4,0.000000e+00,0.000000e+00,2.220446e-16,-2.220446e-16,non-NE,non-NE,False,False,True,True,True,True,True
P-L4-2a,L4,1.065814e-13,1.776357e-15,8.049117e-16,-2.581269e-15,non-NE,non-NE,False,False,True,True,True,True,True
P-L4-2b,L4,1.065814e-13,1.776357e-15,8.049117e-16,-2.581269e-15,non-NE,non-NE,False,False,True,True,True,True,True
P-L4-3a,L4,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,non-NE,non-NE,False,False,True,True,True,True,True
P-L4-3b,L4,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,non-NE,non-NE,False,False,True,True,True,True,True
